# NB6 — Label Noise Audit

**Purpose**: Explore potential label noise in the “ground_truth_binary” column by finding trials where *both* the AI model and the human consensus significantly disagree with the provided ground truth.

In [ ]:
# ── Imports & Setup ──
import sys, os
import numpy as np
import pandas as pd
from scipy import stats

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from helpers import (
    load_and_clean, derive_variables, setup_plotting
)

df = load_and_clean()
df = derive_variables(df)
plt, sns = setup_plotting()

## 1. Grouping by Image
We calculate the total votes and disagreement counts for each unique image.

In [ ]:
# Group by image to analyze per-xray statistics
image_stats = df.groupby('trial_original_image_name').agg(
    ground_truth=('ground_truth_binary', 'first'),
    ai_pred=('ai_prediction', 'first'),
    total_votes=('participant_id', 'count')
).reset_index()

def count_disagreements(image_name):
    trials = df[df['trial_original_image_name'] == image_name]
    gt = trials['ground_truth_binary'].iloc[0]
    disagreements = sum(trials['user_decision'] != gt)
    return disagreements

image_stats['disagree_votes'] = image_stats['trial_original_image_name'].apply(count_disagreements)
image_stats['disagree_ratio'] = image_stats['disagree_votes'] / image_stats['total_votes']
image_stats['human_pred'] = np.where(image_stats['disagree_ratio'] > 0.5, 1 - image_stats['ground_truth'], image_stats['ground_truth'])


## 2. Statistical Testing (Binomial Test)
Null Hypothesis: Humans choose the ground truth label with probability ≥ 0.5. If they systematically choose the incorrect label at a significant rate (p < 0.05), the ground truth is likely noisy.

In [ ]:
def calculate_p_value(row):
    # binomtest(k, n, p) where k=successes (disagreements), n=trials, p=0.5
    res = stats.binomtest(row['disagree_votes'], row['total_votes'], p=0.5, alternative='greater')
    return res.pvalue

image_stats['p_value'] = image_stats.apply(calculate_p_value, axis=1)

# Flag as noise if AI also disagrees with GT AND p_value < 0.05
alpha = 0.05
image_stats['is_noise_candidate'] = (image_stats['ai_pred'] != image_stats['ground_truth']) & (image_stats['p_value'] < alpha)

noise_candidates = image_stats[image_stats['is_noise_candidate']].sort_values('p_value')
print(f"Found {len(noise_candidates)} candidate images with significant label noise.")
display(noise_candidates[['trial_original_image_name', 'ground_truth', 'ai_pred', 'human_pred', 'disagree_votes', 'total_votes', 'disagree_ratio', 'p_value']])

Found 5 candidate images with significant label noise.


,trial_original_image_name,ground_truth,ai_pred,human_pred,disagree_votes,total_votes,disagree_ratio,p_value
1,9023935L.png,0,1,1,103,119,0.865546,4.773317e-17
49,9998089R.png,1,0,0,97,119,0.815126,9.959377e-13
41,9788301L.png,1,0,0,96,119,0.806723,4.267672e-12
26,9360243L.png,0,1,1,91,119,0.764706,2.900987e-09
20,9299531R.png,0,1,1,81,119,0.680672,5.028935e-05


## Findings
These printed `trial_original_image_name` files have an artificially inverted `ground_truth_binary`. The AI model correctly identified them, and the vast majority of human annotators strongly agreed with the AI rather than the assigned ground truth at a statistically significant level.